In [ ]:
import popv
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import random

In [ ]:
ref_adata = sc.read_h5ad('./kidney_snRNA/3f2bf1e8-75a6-4fa2-9f79-dd6bb54529a0.h5ad')
ref_adata

In [ ]:
ref_adata[:5, :].X.toarray()

In [ ]:
Counter(ref_adata.obs['disease'])

In [ ]:
ref_adata_sub = ref_adata[ref_adata.obs['disease']=='normal', :].copy()
Counter(ref_adata_sub.obs['cell_type'])

In [ ]:
len(Counter(ref_adata_sub.obs['donor_id']))

In [ ]:
ref_adata_sub.layers['counts'] = ref_adata_sub.X.copy()

In [ ]:
pd.crosstab(ref_adata_sub.obs['cell_type'], ref_adata_sub.obs['subclass.l1'])

In [ ]:
ref_adata_sub_endo = ref_adata_sub[ref_adata_sub.obs['subclass.l1']=='EC', :].copy()
endo_index = ref_adata_sub_endo.obs_names.to_list()
selected_endo_index = random.sample(endo_index, 1000)

In [ ]:
other_cells_index = ref_adata_sub[ref_adata_sub.obs['subclass.l1']!='EC', :].obs_names.to_list()
len(other_cells_index)

In [ ]:
len(selected_endo_index+other_cells_index)

In [ ]:
sc.pl.umap(ref_adata_sub, color='subclass.l1')

In [ ]:
query_adata = sc.read_h5ad('DBiT_imputation-CellViT-VisiumHD_HumanKidney-8x-spatialDE-all_cells.h5ad')

In [ ]:
query_adata

In [ ]:
query_adata.X = query_adata.layers['counts'].copy()

In [ ]:
query_adata.var_names

In [ ]:
ref_adata_sub.var['feature_id'] = ref_adata_sub.var_names
ref_adata_sub.var_names = np.array(ref_adata_sub.var['feature_name'])

In [ ]:
ref_adata_sub.var_names_make_unique()

In [ ]:
genes_both = np.unique([gene for gene in ref_adata_sub.var_names if gene in query_adata.var_names])
gene_std = np.std(query_adata[:, genes_both].X, axis=0)

In [ ]:
len(genes_both[gene_std!=0])

In [ ]:
len(gene_std)

In [ ]:
np.sum(gene_std<0.01)

In [ ]:
np.sum(gene_std>=0.0269)

In [ ]:
ref_adata_sub_final = ref_adata_sub[:, genes_both[gene_std>0.01]].copy()
query_adata_sub = query_adata[:, genes_both[gene_std>0.01]].copy()

In [ ]:
query_adata_sub.obs['donor_id'] = ['VisiumHD']*query_adata_sub.n_obs

In [ ]:
ref_adata_sub_final.obs['leiden_0.8'] = ['0']* ref_adata_sub_final.n_obs
ref_adata_sub_final.obs['x'] = [0]* ref_adata_sub_final.n_obs
ref_adata_sub_final.obs['y'] = [0]* ref_adata_sub_final.n_obs

In [ ]:
ref_adata_sub_final

In [ ]:
sc.pp.highly_variable_genes(query_adata_sub, n_top_genes=800, flavor='seurat_v3')

In [ ]:
hvgs = query_adata_sub.var_names[query_adata_sub.var['highly_variable']]

In [ ]:
ref_adata_sub_final = ref_adata_sub[:, hvgs].copy()
query_adata_sub = query_adata[:, hvgs].copy()

query_adata_sub.obs['donor_id'] = ['VisiumHD']*query_adata_sub.n_obs

ref_adata_sub_final.obs['leiden_0.8'] = ['0']* ref_adata_sub_final.n_obs
ref_adata_sub_final.obs['x'] = [0]* ref_adata_sub_final.n_obs
ref_adata_sub_final.obs['y'] = [0]* ref_adata_sub_final.n_obs

In [ ]:
sc.pp.calculate_qc_metrics(ref_adata_sub_final, log1p=False, inplace=True)
sc.pp.calculate_qc_metrics(query_adata_sub, log1p=False, inplace=True)

In [ ]:
np.median(ref_adata_sub_final.obs['total_counts'])

In [ ]:
np.median(query_adata_sub.obs['total_counts'])

In [ ]:
#ref_adata_sub_final.X = ref_adata_sub_final.X/np.median(ref_adata_sub_final.obs['total_counts'])*1e4
query_adata_sub.X = query_adata_sub.X/np.median(query_adata_sub.obs['total_counts'])*np.median(ref_adata_sub_final.obs['total_counts'])

In [ ]:
ref_adata_sub_final

In [ ]:
# Following parameters are specific to scRNA-seq ref dataset from cell2location
ref_labels_key='subclass.l1'
ref_batch_key = ['donor_id']

min_celltype_size = np.min(ref_adata_sub_final.obs.groupby('subclass.l1').size())
n_samples_per_label = np.max((min_celltype_size, 100))

#tissue = 'LN_Spleen'
#save_folder = 'popv_results'

# Following parameters are specific to Tabula Sapiens dataset
query_batch_key = ['donor_id']
#methods = ['bbknn','scvi', 'scanvi', 'svm', 'rf', 'scanorama'] #, 'onclass'

# Lesser used parameters
query_labels_key=None
unknown_celltype_label='unknown'

In [ ]:
from popv.preprocessing import Process_Query
from popv.annotation import annotate_data
import random
import anndata

In [ ]:
import os
output_model_fn = f"./VisiumHD_pretrained_model"
if not os.path.exists(output_model_fn):
    os.mkdir(output_model_fn)

In [ ]:
import logging
import os
import subprocess
from typing import List, Optional, Union

import anndata
import numpy as np
import scanpy as sc
import torch
from scanpy._utils import check_nonnegative_integers

from popv import _utils


class Process_Query:
    def __init__(
        self,
        query_adata: anndata.AnnData,
        ref_adata: anndata.AnnData,
        ref_labels_key: str = "cell_ontology_class",
        ref_batch_key: str = "donor_method",
        query_labels_key: Optional[str] = None,
        query_batch_key: Optional[str] = None,
        query_layers_key: Optional[str] = None,
        prediction_mode: Optional[str] = "retrain",
        cl_obo_folder: Optional[Union[List, str, bool]] = None,
        unknown_celltype_label: Optional[str] = "unknown",
        n_samples_per_label: Optional[int] = 300,
        pretrained_scvi_path: Optional[str] = None,
        save_path_trained_models: Optional[str] = "tmp/",
        hvg: Optional[int] = 4000,
        use_gpu: Optional[bool] = True,
        compute_embedding: Optional[bool] = True,
        return_probabilities: Optional[bool] = True,
        n_min_cells = 10
    ) -> None:
        """
        Processes the query and reference dataset in preperation for the annotation pipeline.


        Parameters
        ----------
        query_adata
            AnnData of query cells
        ref_adata
            AnnData of reference cells
        ref_labels_key
            Key in obs field of reference AnnData with cell-type information
        ref_batch_keys
            List of Keys (or None) in obs field of reference AnnData to
            use as batch covariate
        query_labels_key
            Key in obs field of query adata for label information.
            This is only used for training scANVI.
            Make sure to set unknown_celltype_label to mark unlabelled cells.
        query_batch_key
            Key in obs field of query adata for batch information.
        query_layers_key
            If not None, expects raw_count data in query_layers_key.
        prediction_mode
            Execution mode of cell-type annotation.
            "retrain": Train all prediction models and saves them to disk if save_path_trained_models is not None.
            "inference": Classify all cells based on pretrained models.
            "fast": Fast inference using only query cells and single epoch in scArches.
        cl_obo_folder
            Folder containing the cell-type obo for Onclass, ontologies for Onclass and nlp embedding of cell-types.
            Passing a list will use element 1 as obo, element 2 as ontologies and element 3 as nlp embedding.
            Setting it to false will disable ontology use.
        unknown_celltype_label
            If query_labels_key is not None, cells with label unknown_celltype_label
            will be treated as unknown and will be predicted by the model.
        n_samples_per_label
            Reference AnnData will be subset to these amount of cells per cell-type to increase speed.
        pretrained_scvi_path
            If path is None, will train scVI from scratch. Else if
            pretrained_path is set and all the genes in the pretrained models are present
            in query adata, will train the scARCHES version of scVI and scANVI, resulting in
            faster training times.
        save_path_trained_models
            If mode=='retrain' saves models to this directory. Otherwise trained models are expected in this folder.
        hvg
            If Int, subsets data to n highly variable genes according to `sc.pp.highly_variable_genes`
        use_gpu
            If gpu is used to train scVI and scANVI model
        compute_embedding
            Whether UMAP is computed for all integration methods (BBKNN, SCANORAMA, SCANVI, SCVI)
        return_probabilities
            Reports probabilities of the PopV prediction for each method where applicable
        """

        self.labels_key = {"reference": ref_labels_key, "query": query_labels_key}
        self.unknown_celltype_label = unknown_celltype_label
        self.batch_key = {"reference": ref_batch_key, "query": query_batch_key}
        self.n_min = n_min_cells

        if pretrained_scvi_path is None and prediction_mode != "retrain":
            self.pretrained_scvi_path = save_path_trained_models + "/scvi/"
        else:
            self.pretrained_scvi_path = pretrained_scvi_path

        if save_path_trained_models is not None:
            if save_path_trained_models[-1] != "/":
                save_path_trained_models += "/"
            if not os.path.exists(save_path_trained_models):
                os.makedirs(save_path_trained_models)
        self.save_path_trained_models = save_path_trained_models

        self.prediction_mode = prediction_mode
        self.return_probabilities = return_probabilities
        self.genes = None
        if self.prediction_mode == "fast":
            self.genes = torch.load(
                self.pretrained_scvi_path + "model.pt",
                map_location="cpu",
            )["var_names"]
        else:
            if self.pretrained_scvi_path is not None:
                pretrained_scvi_genes = torch.load(
                    self.pretrained_scvi_path + "model.pt",
                    map_location="cpu",
                )["var_names"]
                if self.prediction_mode == "inference":
                    pretrained_scanvi_genes = torch.load(
                        self.save_path_trained_models + "/scanvi/model.pt",
                        map_location="cpu",
                    )["var_names"]
                    assert list(pretrained_scvi_genes) == list(
                        pretrained_scanvi_genes
                    ), "Pretrained SCANVI and SCVI model contain different genes. This is not supported. Check models and retrain."

                    onclass_model = np.load(
                        self.save_path_trained_models + "/OnClass.npz",
                        allow_pickle=True,
                    )
                    assert set(onclass_model["genes"]).issubset(
                        set(pretrained_scanvi_genes)
                    ), "Pretrained SCANVI and OnClass model contain different genes. This is not supported. Retrain OnClass."
                else:
                    if not os.path.exists(self.save_path_trained_models):
                        os.makedirs(self.save_path_trained_models)
                self.genes = list(pretrained_scvi_genes)

        if self.genes is not None:
            assert set(self.genes).issubset(
                set(query_adata.var_names)
            ), "Query dataset misses genes that were used for reference model training. Retrain reference model, set mode='retrain'"
            self.query_adata = query_adata[:, self.genes].copy()
            assert (
                hvg is None
            ), "Highly variable gene selection is not available if using trained reference model."
        else:
            self.query_adata = query_adata.copy()
        if query_layers_key is not None:
            self.query_adata.X = self.query_adata.layers[query_layers_key].copy()

        self.validity_checked = False
        self.hvg = hvg
        self.use_gpu = use_gpu
        if self.prediction_mode == "fast":
            self.n_samples_per_label = None
        else:
            self.n_samples_per_label = n_samples_per_label
        self.compute_embedding = compute_embedding
        
        '''
        if cl_obo_folder is None:
            self.cl_obo_file = (
                os.path.dirname(os.path.dirname(__file__)) + "/ontology/cl.obo"
            )
            self.cl_ontology_file = (
                os.path.dirname(os.path.dirname(__file__)) + "/ontology/cl.ontology"
            )
            self.nlp_emb_file = (
                os.path.dirname(os.path.dirname(__file__))
                + "/ontology/cl.ontology.nlp.emb"
            )
            if not os.path.exists(self.nlp_emb_file):
                subprocess.call(
                    [
                        "tar",
                        "-czf",
                        os.path.dirname(os.path.dirname(__file__))
                        + "/ontology/nlp.emb.tar.gz",
                        "cl.ontology.nlp.emb",
                    ]
                )
        elif cl_obo_folder is False:
            self.cl_obo_file = False
            self.cl_ontology_file = False
            self.nlp_emb_file = False
        elif cl_obo_folder is list:
            self.cl_obo_file = cl_obo_folder[0]
            self.cl_ontology_file = cl_obo_folder[1]
            self.nlp_emb_file = cl_obo_folder[2]
        else:
            self.cl_obo_file = cl_obo_folder + "cl.obo"
            self.cl_ontology_file = cl_obo_folder + "cl.ontology"
            self.nlp_emb_file = cl_obo_folder + "cl.ontology.nlp.emb"
        if self.cl_obo_file:
            try:
                with open(self.cl_obo_file):
                    pass
            except FileNotFoundError:
                raise FileNotFoundError(
                    f"{self.cl_obo_file} doesn't exist. Check that folder exists."
                )
        '''

        self.check_validity_anndata(self.query_adata, "query")
        self.setup_dataset(self.query_adata, "query", add_meta="_query")

        if self.prediction_mode != "fast":
            if self.genes:
                self.ref_adata = ref_adata[:, self.genes].copy()
            else:
                self.ref_adata = ref_adata.copy()
            self.setup_dataset(self.ref_adata, "reference")
            self.check_validity_anndata(self.ref_adata, "reference")

        self.preprocess()

    def check_validity_anndata(self, adata, input_type):
        #assert check_nonnegative_integers(
        #    adata.X
        #), f"Make sure input {input_type} adata contains raw_counts"
        assert len(set(adata.var_names)) == len(
            adata.var_names
        ), f"{input_type} dataset contains multiple genes with same gene name."
        assert adata.n_obs > 0, f"{input_type} anndata has no cells."
        assert adata.n_vars > 0, f"{input_type} anndata has no genes."

    def setup_dataset(self, adata, key, add_meta=""):
        if isinstance(self.batch_key[key], list):
            adata.obs["_batch_annotation"] = (
                adata.obs[self.batch_key[key]].astype(str).sum(1).astype("category")
            )
        elif isinstance(self.batch_key[key], str):
            adata.obs["_batch_annotation"] = adata.obs[self.batch_key[key]]
        else:
            adata.obs["_batch_annotation"] = self.unknown_celltype_label
        adata.obs["_batch_annotation"] = (
            adata.obs["_batch_annotation"].astype(str) + add_meta
        )
        adata.obs["_batch_annotation"] = adata.obs["_batch_annotation"].astype(
            "category"
        )

        adata.obs["_labels_annotation"] = self.unknown_celltype_label
        if self.labels_key[key] is not None:
            adata.obs["_labels_annotation"] = adata.obs[self.labels_key[key]].astype(
                "category"
            )

        # subsample the reference cells used for training certain models
        if key == "reference":
            if self.n_samples_per_label is not None:
                adata.obs["_ref_subsample"] = False
                subsample_idx = _utils.subsample_dataset(
                    adata,
                    self.labels_key[key],
                    n_samples_per_label=self.n_samples_per_label,
                    ignore_label=[self.unknown_celltype_label],
                )
                adata.obs.loc[subsample_idx, "_ref_subsample"] = True
            else:
                adata.obs["_ref_subsample"] = True
        else:
            adata.obs["_ref_subsample"] = False

    def preprocess(self):
        if self.genes is None:
            self.ref_adata = self.ref_adata[
                :, np.intersect1d(self.ref_adata.var_names, self.query_adata.var_names)
            ]
            self.query_adata = self.query_adata[
                :, np.intersect1d(self.ref_adata.var_names, self.query_adata.var_names)
            ]

        if self.prediction_mode == "fast":
            self.adata = self.query_adata
            self.adata.obs["_dataset"] = "query"
        else:
            self.adata = anndata.concat(
                (self.ref_adata, self.query_adata),
                axis=0,
                label="_dataset",
                keys=["ref", "query"],
                join="outer",
                fill_value=self.unknown_celltype_label,
            )

        if self.prediction_mode != "fast":
            # Necessary for BBKNN.
            batch_before_filtering = set(self.adata.obs["_batch_annotation"])
            self.adata = self.adata[
                self.adata.obs["_batch_annotation"].isin(
                    self.adata.obs["_batch_annotation"]
                    .value_counts()[
                        self.adata.obs["_batch_annotation"].value_counts() > 8
                    ]
                    .index
                )
            ].copy()
            difference_batches = (
                set(self.adata.obs["_batch_annotation"]) - batch_before_filtering
            )
            if difference_batches:
                logging.warning(
                    f"The following batches will be excluded from annotation because they have less than 9 cells:{difference_batches}."
                )

            # Sort data based on batch for efficiency downstream during SCANORAMA
            self.adata = self.adata[
                self.adata.obs.sort_values(by="_batch_annotation").index
            ].copy()

            self.adata.obs[self.labels_key["reference"]] = self.adata.obs[
                self.labels_key["reference"]
            ].astype("category")

        # Remove any cell with expression below 10 counts.
        zero_cell_names = self.adata[self.adata.X.sum(1) < self.n_min].obs_names
        self.adata.uns["Filtered_cells"] = list(zero_cell_names)
        sc.pp.filter_cells(self.adata, min_counts=self.n_min, inplace=True)
        if len(zero_cell_names) > 0:
            logging.warning(
                f"The following cells will be excluded from annotation because they have low expression:{zero_cell_names}."
            )

        self.adata.layers["scvi_counts"] = self.adata.X.copy()

        if self.hvg is not None and self.adata.n_vars > self.hvg:
            sc.pp.filter_genes(self.adata, min_counts=20, inplace=True)
            try:
                self.adata.var["highly_variable"] = sc.pp.highly_variable_genes(
                    self.adata[self.adata.obs["_dataset"] == "ref"].copy(),
                    n_top_genes=self.hvg,
                    subset=False,
                    layer="scvi_counts",
                    flavor="seurat_v3",
                    inplace=False,
                    batch_key="_batch_annotation",
                )["highly_variable"]
            except (
                ValueError
            ):  # seurat_v3 tends to error with singularities then use Poisson hvg.
                self.adata.var[
                    "highly_variable"
                ] = sc.experimental.pp.highly_variable_genes(
                    self.adata[self.adata.obs["_dataset"] == "ref"].copy(),
                    n_top_genes=self.hvg,
                    subset=False,
                    layer="scvi_counts",
                    flavor="pearson_residuals",
                    inplace=False,
                    batch_key="_batch_annotation",
                )[
                    "highly_variable"
                ]
            self.adata = self.adata[:, self.adata.var["highly_variable"]].copy()

        #sc.pp.normalize_total(self.adata, target_sum=1e4)
        sc.pp.log1p(self.adata)
        #sc.pp.regress_out(self.adata, ['total_counts'])
        self.adata.layers["scaled_counts"] = self.adata.X.copy()
        if self.prediction_mode != "fast":
            sc.pp.scale(
                self.adata, max_value=10, zero_center=True, layer="scaled_counts"
            )
            self.adata.obsm["X_pca"] = sc.tl.pca(self.adata.layers["scaled_counts"])

        # Store values as default for current popv in adata
        self.adata.uns["unknown_celltype_label"] = self.unknown_celltype_label
        self.adata.uns["_pretrained_scvi_path"] = self.pretrained_scvi_path
        self.adata.uns["_save_path_trained_models"] = self.save_path_trained_models
        self.adata.uns["_prediction_mode"] = self.prediction_mode
        #self.adata.uns["_cl_obo_file"] = self.cl_obo_file
        #self.adata.uns["_cl_ontology_file"] = self.cl_ontology_file
        #self.adata.uns["_nlp_emb_file"] = self.nlp_emb_file
        self.adata.uns["_use_gpu"] = self.use_gpu
        self.adata.uns["_compute_embedding"] = self.compute_embedding
        self.adata.uns["_return_probabilities"] = self.return_probabilities
        self.adata.uns["prediction_keys"] = []


In [ ]:
adata = Process_Query(query_adata_sub,
                      ref_adata_sub_final,
                      #save_folder=save_folder,
                      query_batch_key=query_batch_key,
                      query_labels_key=query_labels_key,
                      unknown_celltype_label=None,
                      save_path_trained_models=output_model_fn,
                      prediction_mode="retrain",
                      ref_labels_key=ref_labels_key, 
                      ref_batch_key=ref_batch_key,
                      n_samples_per_label=n_samples_per_label, use_gpu=False, hvg=1986).adata

In [ ]:
adata

In [ ]:
from harmony import harmonize
from pynndescent import PyNNDescentTransformer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline

In [ ]:
adata.obs['_batch_annotation'] = adata.obs['_dataset']

In [ ]:
adata.uns['_cl_obo_file']=False

In [ ]:
methods = ['knn_on_harmony']
random.seed(42)
annotate_data(adata, methods=methods, save_path=f"./popV_res")

In [ ]:
adata.obsm['X_umap'] = adata.obsm['X_umap_harmony_popv']

In [ ]:
sc.pl.umap(adata, color='_dataset')

In [ ]:
query_data_annotated = adata[adata.obs['_dataset']=='query', :].copy()
query_data_annotated

In [ ]:
podocyte_index = (query_data_annotated[:, 'PODXL'].X.toarray() > np.quantile(query_data_annotated[:, 'PODXL'].X.toarray(), .93)).reshape(1, -1)

In [ ]:
celltype = np.array(query_data_annotated.obs['popv_knn_on_harmony_prediction'])
celltype[query_data_annotated.obs['popv_knn_on_harmony_prediction']=='POD'] = 'EC'
celltype[np.where(podocyte_index)[1]] = 'POD'

In [ ]:
query_data_annotated.obs['celltype_final'] = celltype

In [ ]:
Counter(query_data_annotated.obs['celltype_final'])

In [ ]:
sample_id = 'Visium_HD_Human_Kidney_FFPE_all_cells'
query_data_annotated.uns = {}
query_data_annotated.uns['spatial'] = {}
query_data_annotated.uns['spatial'][sample_id] = {}
query_data_annotated.uns['spatial'][sample_id]['images'] = {}
#query_data_annotated_query.uns['spatial'][sample_id]['images']['lowres'] = imageLN_flip # rotated_imageLN
query_data_annotated.uns['spatial'][sample_id]['scalefactors'] = {}
query_data_annotated.uns['spatial'][sample_id]['scalefactors']['tissue_hires_scalef'] = 0.13636674  # 78.5
query_data_annotated.uns['spatial'][sample_id]['scalefactors']['spot_diameter_fullres'] =  233.6586019512961 / 8 #127.8358082731091 #2.18
query_data_annotated.uns['spatial'][sample_id]['scalefactors']['fiducial_diameter_fullres'] = 1204.802166311371
query_data_annotated.uns['spatial'][sample_id]['scalefactors']['tissue_lowres_scalef'] = 0.013636674 # 78.5
query_data_annotated.obsm['spatial'] = np.array([query_data_annotated.obs['x'], query_data_annotated.obs['y']]).T

In [ ]:
sc.pl.spatial(query_data_annotated, color='popv_knn_on_harmony_prediction', size=1.5, img_key=None)

In [ ]:
sc.pl.spatial(query_data_annotated, color='celltype_final', size=1.5, img_key=None)

In [ ]:
ref_data_annotated = adata[adata.obs['_dataset']=='ref', :].copy()
ref_data_annotated.obs['celltype_final'] = ref_data_annotated.obs['popv_knn_on_harmony_prediction']

In [ ]:
full_data_annotated = sc.concat((ref_data_annotated, query_data_annotated))

In [ ]:
full_data_annotated.obs = full_data_annotated.obs.drop(columns=['is_primary_data'])
query_data_annotated.obs = query_data_annotated.obs.drop(columns=['is_primary_data'])

In [ ]:
full_data_annotated.write_h5ad('VisiumHD_withref_annoated_800.h5ad')

In [ ]:
query_data_annotated.write_h5ad('VisiumHD_NOTwithref_annoated.h5ad')

In [ ]:
query_adata_sub = query_adata_sub[query_data_annotated.obs_names, :]

In [ ]:
query_data_annotated.layers['counts'] = query_adata_sub.layers['counts'].copy()

In [ ]:
query_data_annotated.write_h5ad('VisiumHD_NOTwithref_annoated.h5ad')

In [ ]:
query_data_annotated_sub = query_data_annotated[(query_data_annotated.obs['celltype_final']!='NEU')*(query_data_annotated.obs['celltype_final']!='PapE'), :].copy()

In [ ]:
#query_data_annotated.X = query_data_annotated.layers['counts'].copy()
#sc.pp.normalize_total(query_data_annotated, target_sum=1e4)
#sc.pp.log1p(query_data_annotated)
sc.tl.rank_genes_groups(query_data_annotated, groupby='celltype_final', method='wilcoxon')

In [ ]:
np.unique(query_data_annotated.obs['celltype_final'])

In [ ]:
clusters_order = ['POD', 'PT', 'TAL', 'PC', 'CNT', 'DCT', 'DTL', 'IC', 'PEC', 'PapE', 'EC', 'FIB', 'IMM', 
       'VSM/P']
lg_list = []
lg_list_group = []
for lk in clusters_order:
    genes = list(sc.get.rank_genes_groups_df(query_data_annotated, group=lk, pval_cutoff=0.05, log2fc_min=1.5).names.values)
    lg_list += genes
    lg_list_group.append(genes)

In [ ]:
len(lg_list_group[0])

In [ ]:
clusters_order

In [ ]:
def grouped_obs_mean(adata, group_key, layer=None, gene_symbols=None):
    if layer is not None:
        getX = lambda x: x.layers[layer]
    else:
        getX = lambda x: x.X
    if gene_symbols is not None:
        new_idx = adata.var[idx]
    else:
        new_idx = adata.var_names

    grouped = adata.obs.groupby(group_key)
    out = pd.DataFrame(
        np.zeros((adata.shape[1], len(grouped)), dtype=np.float64),
        columns=list(grouped.groups.keys()),
        index=adata.var_names
    )

    for group, idx in grouped.indices.items():
        print(group)
        print(idx)
        X = getX(adata[idx])
        out[group] = np.ravel(X.mean(axis=0, dtype=np.float64))
    return out

In [ ]:
query_data_annotated

In [ ]:
from collections import Counter
Counter(query_data_annotated.obs['celltype_final'])

In [ ]:
query_data_annotated.X

In [ ]:
grouped = query_data_annotated.obs.groupby('celltype_final')
out = pd.DataFrame(
    np.zeros((query_data_annotated.shape[1], len(grouped)), dtype=np.float64),
    columns=list(grouped.groups.keys()),
    index=query_data_annotated.var_names
)

for group, idx in grouped.indices.items():
    X = query_data_annotated[idx].X
    out[group] = np.ravel(X.mean(axis=0, dtype=np.float64))

In [ ]:
gene_list_len = np.cumsum([len(list) for list in lg_list_group])

In [ ]:
import seaborn as sns
from scipy.stats import zscore
fig, axes = plt.subplots(figsize=(30, 6), nrows=1)
#sns.heatmap(xenium_cluster_mean.loc[lg_list, clusters_order].T.apply(zscore), cmap="inferno_r", ax=axes[0], xticklabels = 3)
sns.heatmap(out.loc[lg_list, clusters_order].T.apply(zscore), cmap="inferno_r", xticklabels = True)

# Hide grid lines
axes.grid(False)
axes.set_xticks(gene_list_len, clusters_order)

axes.set_yticklabels(axes.get_yticklabels(), size=25, fontweight='bold')
axes.set_xticklabels(axes.get_xticklabels(), size=25, fontweight='bold')

axes.collections[0].colorbar.ax.tick_params(labelsize=20)

### subsampling Kidney data

In [ ]:
query_adata_sub

In [ ]:
random.seed(42)
query_index = query_adata_sub.obs_names.to_list()
selected_index = random.sample(query_index, 25000)

In [ ]:
query_adata_sub_sub = query_adata_sub[selected_index, :].copy()
query_adata_sub_sub

In [ ]:
sampled_adata = Process_Query(query_adata_sub_sub,
                      ref_adata_sub_final,
                      #save_folder=save_folder,
                      query_batch_key=query_batch_key,
                      query_labels_key=query_labels_key,
                      unknown_celltype_label=None,
                      save_path_trained_models=output_model_fn,
                      prediction_mode="retrain",
                      ref_labels_key=ref_labels_key, 
                      ref_batch_key=ref_batch_key,
                      n_samples_per_label=n_samples_per_label, use_gpu=False, hvg=1986).adata

In [ ]:
from harmony import harmonize
from pynndescent import PyNNDescentTransformer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline

sampled_adata.obs['_batch_annotation'] = sampled_adata.obs['_dataset']

sampled_adata.uns['_cl_obo_file']=False

methods = ['knn_on_harmony']
random.seed(42)
annotate_data(sampled_adata, methods=methods, save_path=f"./popV_res")

In [ ]:
sampled_query_data_annotated = sampled_adata[sampled_adata.obs['_dataset']=='query', :].copy()
sampled_query_data_annotated

podocyte_index = (sampled_query_data_annotated[:, 'PODXL'].X.toarray() > np.quantile(sampled_query_data_annotated[:, 'PODXL'].X.toarray(), .93)).reshape(1, -1)

celltype = np.array(sampled_query_data_annotated.obs['popv_knn_on_harmony_prediction'])
#celltype[query_data_annotated.obs['popv_knn_on_harmony_prediction']=='POD'] = 'EC'
celltype[np.where(podocyte_index)[1]] = 'POD'

sampled_query_data_annotated.obs['celltype_final'] = celltype

In [ ]:
ref_data_annotated = sampled_adata[sampled_adata.obs['_dataset']=='ref', :].copy()
ref_data_annotated.obs['celltype_final'] = ref_data_annotated.obs['subclass.l1']

full_data_annotated = sc.concat((ref_data_annotated, sampled_query_data_annotated))

#full_data_annotated.obs = full_data_annotated.obs.drop(columns=['is_primary_data'])
#sampled_query_data_annotated.obs = sampled_query_data_annotated.obs.drop(columns=['is_primary_data'])
dataset = full_data_annotated.obs['_dataset'].astype(str)
dataset[dataset=='ref']='scRNA-seq Ref'
dataset[dataset=='query']='SuperFocus'
full_data_annotated.obs['data_group']=dataset

full_data_annotated.obsm['X_pca'] = full_data_annotated.obsm['X_pca_harmony']
full_data_annotated.obsm['X_umap'] = full_data_annotated.obsm['X_umap_harmony_popv']
sc.pl.umap(ref_data_annotated[ref_data_annotated.obs['celltype_final']!='NEU', :], color='celltype_final', palette=palette2)
sc.pl.umap(full_data_annotated[full_data_annotated.obs['celltype_final']!='NEU', :], color='data_group')
sc.pl.umap(full_data_annotated[full_data_annotated.obs['celltype_final']!='NEU', :], color='celltype_final', palette=palette2)


In [ ]:
sc.pl.umap(sampled_query_data_annotated, color='celltype_final')